# 02 — Prototipe & Validasi MCDM (AHP + TOPSIS vs SAW & WP): LapPick

**Tujuan Notebook:**
1. Memuat `laptops_clean.csv` (949 laptop bersih dengan perbaikan `refresh_rate_hz`, atau otomatis membentuknya dari `kagglehub` jika belum diunggah).
2. Menyusun matriks perbandingan berpasangan **AHP (9 Kriteria: C1–C9)** untuk 4 preset profil pengguna (**Mahasiswa, Gamer, Kreator, Kantoran**) dan memverifikasi **Consistency Ratio ($CR \le 0.10$)**.
3. Mengimplementasikan perhitungan **TOPSIS** (utama) serta **SAW** dan **WP** (pembanding validasi).
4. Menghitung korelasi peringkat **Spearman ($\rho$)** dan irisan Top-5/Top-10 antar metode.
5. Menjalankan **Uji Sensitivitas Bobot** ($\pm 10\%$ dan $\pm 20\%$) untuk menguji stabilitas keputusan.
6. Mengekspor `preset_profiles.json`, `topsis_fixture.json` (acuan *cross-check* PHP Laravel $\leftrightarrow$ Python), dan `mcdm_validation_summary.json` ke dalam `lappick_colab_02_outputs.zip`.

In [1]:
# === SEL 1: IMPORT & DEFINISI 9 KRITERIA ===
import re
import json
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

OUTPUT_DIR = Path('output_02')
OUTPUT_DIR.mkdir(exist_ok=True)

CRITERIA = [
    {'code': 'C1', 'name_id': 'Harga',               'name_en': 'Price',            'type': 'cost',    'unit': 'Rp',   'col': 'price_idr'},
    {'code': 'C2', 'name_id': 'Kapasitas RAM',       'name_en': 'RAM Capacity',     'type': 'benefit', 'unit': 'GB',   'col': 'ram_gb'},
    {'code': 'C3', 'name_id': 'Storage Efektif',     'name_en': 'Effective Storage','type': 'benefit', 'unit': 'GB',   'col': 'storage_effective_gb'},
    {'code': 'C4', 'name_id': 'Peringkat CPU',       'name_en': 'CPU Ranking',      'type': 'cost',    'unit': 'rank', 'col': 'cpu_ranking'},
    {'code': 'C5', 'name_id': 'Skor Benchmark GPU',  'name_en': 'GPU Benchmark',    'type': 'benefit', 'unit': 'pts',  'col': 'gpu_benchmark'},
    {'code': 'C6', 'name_id': 'Daya Tahan Baterai',  'name_en': 'Battery Life',     'type': 'benefit', 'unit': 'jam',  'col': 'battery_hours'},
    {'code': 'C7', 'name_id': 'Bobot Laptop',        'name_en': 'Weight',           'type': 'cost',    'unit': 'kg',   'col': 'weight_kg'},
    {'code': 'C8', 'name_id': 'Rating Pengguna',     'name_en': 'User Rating',      'type': 'benefit', 'unit': '/5',   'col': 'user_rating'},
    {'code': 'C9', 'name_id': 'Refresh Rate Layar',  'name_en': 'Refresh Rate',     'type': 'benefit', 'unit': 'Hz',   'col': 'refresh_rate_hz'},
]

CRITERIA_CODES = [c['code'] for c in CRITERIA]
CRITERIA_COLS = [c['col'] for c in CRITERIA]
CRITERIA_TYPES = [c['type'] for c in CRITERIA]
print(f'Terdefinisi {len(CRITERIA)} kriteria:', CRITERIA_CODES)

Terdefinisi 9 kriteria: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9']


In [2]:
# === SEL 2: MUAT laptops_clean.csv (ATAU BANGUN OTOMATIS DARI KAGGLEHUB) ===
# Termasuk perbaikan bug digit pertama Refresh Rate dari judul laptop

def fix_refresh_rate_from_name(df_in):
    df_out = df_in.copy()
    rr_map = {1: 144, 2: 240, 3: 300, 6: 60, 9: 90}
    fixed_vals = []
    for _, row in df_out.iterrows():
        m = re.search(r'(\d{2,3})\s*hz', str(row['name']), flags=re.I)
        if m:
            fixed_vals.append(int(m.group(1)))
        else:
            v = int(float(row['refresh_rate_hz']))
            fixed_vals.append(rr_map.get(v, v if v >= 60 else 60))
    df_out['refresh_rate_hz'] = fixed_vals
    return df_out

clean_candidates = list(Path('.').rglob('laptops_clean.csv'))
if clean_candidates:
    df = pd.read_csv(clean_candidates[0])
    df = fix_refresh_rate_from_name(df)
    print(f'Memuat laptops_clean.csv dari {clean_candidates[0]} ({len(df)} baris)')
else:
    print('laptops_clean.csv belum ada di direktori kerja, membangun otomatis dari kagglehub...')
    import kagglehub
    dl_dir = Path(kagglehub.dataset_download('dhanushbommavaram/laptop-ranked-dataset'))
    raw_csv = list(dl_dir.rglob('*.csv'))[0]
    df_raw = pd.read_csv(raw_csv)
    COMPANY_MAP = {1:'ASUS',2:'HP',3:'Lenovo',4:'Dell',5:'MSI',6:'Realme',7:'Avita',8:'Acer',9:'Samsung',10:'Infinix',11:'LG',12:'Apple',13:'Nokia',14:'RedmiBook',15:'Mi',16:'VAIO'}
    TYPE_MAP = {1:'Gaming Laptop',2:'Thin and Light Laptop',3:'2 in 1 Laptop',4:'Notebook',5:'Laptop',6:'2 in 1 Gaming Laptop',7:'Business Laptop',8:'Chromebook',9:'Creator Laptop'}
    PROC_BRAND_MAP = {1:'Intel',2:'AMD',3:'Qualcomm',4:'Apple',5:'MediaTek'}
    OS_MAP = {1:'Windows',2:'Chrome OS',3:'DOS',4:'macOS',5:'Ubuntu'}
    df = pd.DataFrame()
    df['name'] = df_raw['name'].astype(str).str.strip()
    df['brand'] = df_raw['company'].map(COMPANY_MAP).fillna('Unknown')
    df['type_name'] = df_raw['Type'].map(TYPE_MAP).fillna('Laptop')
    df['os'] = df_raw['Operating System'].map(OS_MAP).fillna('Windows')
    df['processor_brand'] = df_raw['Processor Brand'].map(PROC_BRAND_MAP).fillna('Unknown')
    df['cpu_name'] = df_raw['Processor name'].astype(str).str.strip()
    df['gpu_name'] = df_raw[[c for c in df_raw.columns if 'gpu name' in c.lower()][0]].astype(str).str.strip()
    df['ram_type'] = df_raw['RAM Type'].astype(str).str.strip()
    df['ram_type_score'] = pd.to_numeric(df_raw['ram_type_tokenized'], errors='coerce').fillna(4).astype(int)
    df['has_ssd'] = (pd.to_numeric(df_raw['SSD'], errors='coerce').fillna(1) > 0).astype(int)
    df['expandable_memory'] = (pd.to_numeric(df_raw['Expandable Memory'], errors='coerce').fillna(0) > 0).astype(int)
    df['touchscreen'] = (pd.to_numeric(df_raw['Touchscreen'], errors='coerce').fillna(0) > 0).astype(int)
    df['dedicated_vram_gb'] = pd.to_numeric(df_raw['Dedicated Graphic Memory Capacity'], errors='coerce').fillna(0.0)
    df['screen_inches'] = pd.to_numeric(df_raw['Screen Size (in inch)'], errors='coerce').fillna(15.6)
    df['screen_resolution'] = df_raw['screen_resolution'].astype(str).str.strip()
    df['source_link'] = df_raw['link'].astype(str).str.strip()
    df['price_inr'] = pd.to_numeric(df_raw['Price (in Indian Rupees)'], errors='coerce')
    df['price_idr'] = (df['price_inr'] * 190).round(0).astype(np.int64)
    df['ram_gb'] = pd.to_numeric(df_raw['RAM (in GB)'], errors='coerce').astype(int)
    raw_st = pd.to_numeric(df_raw['Storage'], errors='coerce')
    df['storage_gb'] = np.where(raw_st <= 8, raw_st * 1024, raw_st).astype(int)
    df['storage_effective_gb'] = np.where(df['has_ssd'] == 1, df['storage_gb'] * 1.0, df['storage_gb'] * 0.6)
    df['cpu_ranking'] = pd.to_numeric(df_raw['CPU_ranking'], errors='coerce').astype(int)
    df['gpu_benchmark'] = pd.to_numeric(df_raw['gpu_benchmark'], errors='coerce')
    df['battery_hours'] = pd.to_numeric(df_raw['battery_backup'], errors='coerce')
    df['weight_kg'] = pd.to_numeric(df_raw['Weight (in kg)'], errors='coerce')
    df['user_rating'] = pd.to_numeric(df_raw['user rating'], errors='coerce')
    df['refresh_rate_hz'] = pd.to_numeric(df_raw['Refresh Rate'], errors='coerce').fillna(60).astype(int)
    df = df.drop_duplicates(subset=['name', 'price_inr', 'ram_gb', 'storage_gb', 'cpu_name']).copy()
    df['is_imputed'] = np.where(df['user_rating'].isna(), 1, 0)
    med_by_type = df.groupby('type_name')['user_rating'].transform('median')
    df['user_rating'] = df['user_rating'].fillna(med_by_type).fillna(df['user_rating'].median())
    df = fix_refresh_rate_from_name(df).reset_index(drop=True)
    df.insert(0, 'id', np.arange(1, len(df) + 1))

print('Distribusi refresh_rate_hz:', df['refresh_rate_hz'].value_counts().sort_index().to_dict())
df.head(3)

Memuat laptops_clean.csv dari laptops_clean.csv (949 baris)
Distribusi refresh_rate_hz: {60: 845, 90: 4, 120: 31, 144: 51, 165: 9, 240: 3, 300: 5, 360: 1}


,id,name,brand,type_name,os,processor_brand,cpu_name,gpu_name,ram_type,ram_type_score,has_ssd,expandable_memory,touchscreen,dedicated_vram_gb,screen_inches,screen_resolution,source_link,price_inr,price_idr,ram_gb,storage_gb,storage_effective_gb,cpu_ranking,gpu_benchmark,battery_hours,weight_kg,user_rating,refresh_rate_hz,is_imputed
0,1,asus rog strix scar 17 core i9 12th gen - (32 ...,ASUS,Gaming Laptop,Windows,Intel,intel core i9 12th gen,nvidia geforce rtx 3070 ti,ddr5,7,1,1,0,8.0,17.3,1440,https://www.flipkart.com/asus-rog-strix-scar-1...,234990,44648100,32,1000,1000.0,4,168.0,4.0,2.90,5.0,60,0
1,2,asus rog strix scar 15 core i9 12th gen - (32 ...,ASUS,Gaming Laptop,Windows,Intel,intel core i9 12th gen,nvidia geforce rtx 3070 ti,ddr5,7,1,1,0,8.0,15.6,1440,https://www.flipkart.com/asus-rog-strix-scar-1...,229990,43698100,32,1000,1000.0,4,168.0,4.0,2.30,4.5,60,1
2,3,hp victus ryzen 7 octa core 5800h - (16 gb/512...,HP,Gaming Laptop,Windows,AMD,amd ryzen 7 octa core,nvidia geforce rtx 3050,ddr4,4,1,0,0,4.0,16.1,1080,https://www.flipkart.com/hp-victus-ryzen-7-oct...,104091,19777290,16,512,512.0,64,72.6,4.0,2.48,4.5,60,1


In [3]:
# === SEL 3: ENGINE AHP & PENYUSUNAN 4 PRESET PROFIL (CR <= 0.10) ===
RI_TABLE = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}

def compute_ahp(pairwise_matrix):
    """
    Menghitung bobot prioritas AHP (metode normalisasi kolom + rata-rata baris),
    lambda_max, CI, RI, dan CR sesuai spesifikasi di 03_ARCHITECTURE_SPK_LAPTOP.md.
    """
    A = np.array(pairwise_matrix, dtype=float)
    n = A.shape[0]
    col_sums = A.sum(axis=0)
    norm_matrix = A / col_sums
    weights = norm_matrix.mean(axis=1)
    weights = weights / weights.sum()  # Pastikan tepat berjumlah 1.0

    Aw = A @ weights
    lambda_vec = Aw / weights
    lambda_max = float(np.mean(lambda_vec))
    ci = float((lambda_max - n) / (n - 1)) if n > 1 else 0.0
    ri = RI_TABLE.get(n, 1.49)
    cr = float(ci / ri) if ri > 0 else 0.0

    return {
        'n': n,
        'weights': weights,
        'normalized_matrix': norm_matrix,
        'lambda_max': lambda_max,
        'ci': ci,
        'ri': ri,
        'cr': cr,
        'is_consistent': bool(cr <= 0.10)
    }

def build_saaty_pairwise_from_scores(importance_scores):
    """
    Membentuk matriks perbandingan berpasangan skala Saaty (1..9 dan 1/9..1)
    dari skor tingkat kepentingan (1..9) setiap kriteria sehingga dijamin konsisten (CR << 0.10).
    Jika score_i >= score_j, a_ij = min(9, max(1, round(score_i - score_j + 1))).
    """
    n = len(importance_scores)
    M = np.ones((n, n), dtype=float)
    for i in range(n):
        for j in range(i + 1, n):
            diff = importance_scores[i] - importance_scores[j]
            if diff >= 0:
                scale = float(min(9, max(1, int(round(diff + 1)))))
                M[i, j] = scale
                M[j, i] = 1.0 / scale
            else:
                scale = float(min(9, max(1, int(round(-diff + 1)))))
                M[i, j] = 1.0 / scale
                M[j, i] = scale
    return M

# Skor kepentingan (skala 1..8) untuk C1..C9 per profil:
# Urutan kode: [C1 Harga, C2 RAM, C3 Storage, C4 CPU, C5 GPU, C6 Baterai, C7 Bobot, C8 Rating, C9 RefreshRate]
PROFILE_DEFINITIONS = [
    {
        'slug': 'student',
        'name_id': 'Mahasiswa / Pelajar',
        'name_en': 'Student',
        'description_id': 'Mengutamakan harga terjangkau, bobot ringan untuk dibawa kuliah, dan daya tahan baterai awet.',
        'description_en': 'Prioritizes affordable price, lightweight portability for campus, and long battery life.',
        'importance_scores': [8, 5, 4, 4, 2, 6, 7, 4, 1]
    },
    {
        'slug': 'gamer',
        'name_id': 'Gamer & Esports',
        'name_en': 'Gamer & Esports',
        'description_id': 'Mengutamakan performa GPU, peringkat CPU tinggi, layar refresh rate tinggi (120-360Hz), dan RAM besar.',
        'description_en': 'Prioritizes GPU benchmark, high-tier CPU, high refresh rate display (120-360Hz), and large RAM.',
        'importance_scores': [3, 6, 5, 7, 8, 2, 1, 4, 7]
    },
    {
        'slug': 'creator',
        'name_id': 'Kreator Konten & Desainer',
        'name_en': 'Content Creator & Designer',
        'description_id': 'Mengutamakan kekuatan CPU & GPU untuk rendering, kapasitas RAM & SSD besar, serta layar mulus.',
        'description_en': 'Prioritizes CPU & GPU power for rendering, large RAM & SSD storage, and smooth display.',
        'importance_scores': [3, 7, 6, 8, 7, 4, 3, 5, 5]
    },
    {
        'slug': 'office',
        'name_id': 'Pekerja Kantoran & Bisnis',
        'name_en': 'Office & Business Professional',
        'description_id': 'Mengutamakan mobilitas (bobot ringan & baterai tahan lama), keandalan (rating tinggi), CPU responsif, dan harga rasional.',
        'description_en': 'Prioritizes mobility (lightweight & battery life), reliability (user rating), responsive CPU, and reasonable price.',
        'importance_scores': [6, 5, 4, 6, 2, 8, 7, 6, 2]
    }
]

preset_profiles_output = []
for prof in PROFILE_DEFINITIONS:
    M = build_saaty_pairwise_from_scores(prof['importance_scores'])
    res = compute_ahp(M)
    assert res['is_consistent'], f"Profil {prof['slug']} tidak konsisten! CR={res['cr']}"
    weights_dict = {code: round(float(w), 6) for code, w in zip(CRITERIA_CODES, res['weights'])}
    preset_profiles_output.append({
        'slug': prof['slug'],
        'name_id': prof['name_id'],
        'name_en': prof['name_en'],
        'description_id': prof['description_id'],
        'description_en': prof['description_en'],
        'pairwise_matrix': np.round(M, 6).tolist(),
        'weights': weights_dict,
        'lambda_max': round(res['lambda_max'], 6),
        'ci': round(res['ci'], 6),
        'ri': res['ri'],
        'cr': round(res['cr'], 6),
        'is_consistent': res['is_consistent']
    })
    print(f"[{prof['slug']:8s}] lambda_max={res['lambda_max']:.4f} | CI={res['ci']:.4f} | CR={res['cr']:.4f} (Konsisten: {res['is_consistent']})")
    print('   Bobot:', weights_dict)


[student ] lambda_max=9.2848 | CI=0.0356 | CR=0.0245 (Konsisten: True)
   Bobot: {'C1': 0.303031, 'C2': 0.098482, 'C3': 0.062618, 'C4': 0.062618, 'C5': 0.029471, 'C6': 0.146851, 'C7': 0.212989, 'C8': 0.062618, 'C9': 0.021323}
[gamer   ] lambda_max=9.3135 | CI=0.0392 | CR=0.0270 (Konsisten: True)
   Bobot: {'C1': 0.040334, 'C2': 0.12346, 'C3': 0.084824, 'C4': 0.184096, 'C5': 0.276109, 'C6': 0.028154, 'C7': 0.020465, 'C8': 0.05846, 'C9': 0.184096}
[creator ] lambda_max=9.1772 | CI=0.0221 | CR=0.0153 (Konsisten: True)
   Bobot: {'C1': 0.030783, 'C2': 0.1785, 'C3': 0.115488, 'C4': 0.271615, 'C5': 0.1785, 'C6': 0.04711, 'C7': 0.030783, 'C8': 0.073611, 'C9': 0.073611}
[office  ] lambda_max=9.2194 | CI=0.0274 | CR=0.0189 (Konsisten: True)
   Bobot: {'C1': 0.118033, 'C2': 0.075161, 'C3': 0.05088, 'C4': 0.118033, 'C5': 0.025112, 'C6': 0.280981, 'C7': 0.188656, 'C8': 0.118033, 'C9': 0.025112}


In [4]:
# === SEL 4: ENGINE TOPSIS, SAW, DAN WP ===
def compute_topsis(df_items, weights_vec, criteria=CRITERIA):
    """
    Menghitung perankingan TOPSIS sesuai rumus di 03_ARCHITECTURE_SPK_LAPTOP.md:
    1. Normalisasi vektor r_ij = x_ij / sqrt(sum(x_ij^2))
    2. Matriks terbobot y_ij = w_j * r_ij
    3. Solusi ideal positif (A+) & negatif (A-)
    4. Jarak Euclidean D+ & D-
    5. Nilai preferensi C_i = D- / (D+ + D-)
    """
    cols = [c['col'] for c in criteria]
    types = [c['type'] for c in criteria]
    X = df_items[cols].to_numpy(dtype=float)
    w = np.array(weights_vec, dtype=float)
    w = w / w.sum()

    denom = np.sqrt((X ** 2).sum(axis=0))
    denom = np.where(denom == 0, 1e-12, denom)
    R = X / denom
    Y = R * w

    ideal_pos = np.zeros(len(criteria), dtype=float)
    ideal_neg = np.zeros(len(criteria), dtype=float)
    for j, t in enumerate(types):
        if t == 'benefit':
            ideal_pos[j] = Y[:, j].max()
            ideal_neg[j] = Y[:, j].min()
        else:
            ideal_pos[j] = Y[:, j].min()
            ideal_neg[j] = Y[:, j].max()

    d_plus = np.sqrt(((Y - ideal_pos) ** 2).sum(axis=1))
    d_minus = np.sqrt(((Y - ideal_neg) ** 2).sum(axis=1))
    scores = d_minus / np.where((d_plus + d_minus) == 0, 1e-12, (d_plus + d_minus))

    out = df_items[['id', 'name', 'brand', 'type_name', 'price_idr', 'ram_gb', 'storage_gb', 'cpu_name', 'gpu_name', 'refresh_rate_hz']].copy()
    out['d_plus'] = d_plus
    out['d_minus'] = d_minus
    out['topsis_score'] = scores
    out = out.sort_values(by=['topsis_score', 'id'], ascending=[False, True]).reset_index(drop=True)
    out['rank'] = np.arange(1, len(out) + 1)
    return {
        'ranking_df': out,
        'denom': denom,
        'R': R,
        'Y': Y,
        'ideal_pos': ideal_pos,
        'ideal_neg': ideal_neg,
        'd_plus': d_plus,
        'd_minus': d_minus,
        'scores': scores
    }

def compute_saw(df_items, weights_vec, criteria=CRITERIA):
    cols = [c['col'] for c in criteria]
    types = [c['type'] for c in criteria]
    X = df_items[cols].to_numpy(dtype=float)
    w = np.array(weights_vec, dtype=float)
    w = w / w.sum()
    R_saw = np.zeros_like(X)
    for j, t in enumerate(types):
        if t == 'benefit':
            R_saw[:, j] = X[:, j] / max(X[:, j].max(), 1e-12)
        else:
            R_saw[:, j] = max(X[:, j].min(), 1e-12) / np.where(X[:, j] == 0, 1e-12, X[:, j])
    scores = R_saw @ w
    return scores

def compute_wp(df_items, weights_vec, criteria=CRITERIA):
    cols = [c['col'] for c in criteria]
    types = [c['type'] for c in criteria]
    X = df_items[cols].to_numpy(dtype=float)
    w = np.array(weights_vec, dtype=float)
    w = w / w.sum()
    signed_w = np.array([wj if t == 'benefit' else -wj for wj, t in zip(w, types)])
    # Gunakan log-sum untuk stabilitas numerik: log(S_i) = sum(signed_w_j * log(x_ij))
    log_S = (np.log(np.maximum(X, 1e-12)) * signed_w).sum(axis=1)
    S = np.exp(log_S - log_S.max())
    V = S / S.sum()
    return V


In [5]:
# === SEL 5: PERANKINGAN 4 PROFIL & VALIDASI SILANG (TOPSIS vs SAW vs WP) ===
profile_rankings_summary = {}

for prof in preset_profiles_output:
    slug = prof['slug']
    w_vec = [prof['weights'][c] for c in CRITERIA_CODES]
    topsis_res = compute_topsis(df, w_vec)
    saw_scores = compute_saw(df, w_vec)
    wp_scores = compute_wp(df, w_vec)

    # Gabungkan skor berdasarkan urutan asli df
    comp_df = df[['id', 'name', 'brand', 'type_name', 'price_idr', 'ram_gb', 'storage_gb', 'cpu_ranking', 'gpu_benchmark', 'battery_hours', 'weight_kg', 'user_rating', 'refresh_rate_hz']].copy()
    comp_df['topsis_score'] = topsis_res['scores']
    comp_df['saw_score'] = saw_scores
    comp_df['wp_score'] = wp_scores
    comp_df['rank_topsis'] = comp_df['topsis_score'].rank(ascending=False, method='min').astype(int)
    comp_df['rank_saw'] = comp_df['saw_score'].rank(ascending=False, method='min').astype(int)
    comp_df['rank_wp'] = comp_df['wp_score'].rank(ascending=False, method='min').astype(int)

    rho_saw, _ = spearmanr(comp_df['topsis_score'], comp_df['saw_score'])
    rho_wp, _ = spearmanr(comp_df['topsis_score'], comp_df['wp_score'])

    top10_topsis = set(comp_df.nsmallest(10, 'rank_topsis')['id'])
    top10_saw = set(comp_df.nsmallest(10, 'rank_saw')['id'])
    top10_wp = set(comp_df.nsmallest(10, 'rank_wp')['id'])

    top5_records = topsis_res['ranking_df'].head(5)[['rank', 'id', 'brand', 'name', 'type_name', 'price_idr', 'ram_gb', 'storage_gb', 'refresh_rate_hz', 'd_plus', 'd_minus', 'topsis_score']].to_dict(orient='records')
    profile_rankings_summary[slug] = {
        'spearman_topsis_vs_saw': round(float(rho_saw), 4),
        'spearman_topsis_vs_wp': round(float(rho_wp), 4),
        'top10_overlap_saw': int(len(top10_topsis & top10_saw)),
        'top10_overlap_wp': int(len(top10_topsis & top10_wp)),
        'top5_recommendations': top5_records
    }

    print(f'\n================ PROFIL: {slug.upper()} ================')
    print(f'Spearman TOPSIS vs SAW = {rho_saw:.4f} (Irisan Top-10: {len(top10_topsis & top10_saw)}/10)')
    print(f'Spearman TOPSIS vs WP  = {rho_wp:.4f} (Irisan Top-10: {len(top10_topsis & top10_wp)}/10)')
    display(topsis_res['ranking_df'].head(5)[['rank', 'id', 'brand', 'type_name', 'price_idr', 'ram_gb', 'storage_gb', 'refresh_rate_hz', 'topsis_score']])



================ PROFIL: STUDENT ================
Spearman TOPSIS vs SAW = 0.6778 (Irisan Top-10: 0/10)
Spearman TOPSIS vs WP  = 0.6485 (Irisan Top-10: 5/10)


,rank,id,brand,type_name,price_idr,ram_gb,storage_gb,refresh_rate_hz,topsis_score
0,1,326,HP,Notebook,7596200,8,1000,60,0.787045
1,2,499,Nokia,Thin and Light Laptop,9498100,16,512,60,0.780011
2,3,15,ASUS,Notebook,7020310,8,256,60,0.776777
3,4,157,Infinix,Thin and Light Laptop,9688100,16,512,60,0.775715
4,5,160,Infinix,Thin and Light Laptop,9688100,16,512,60,0.775715



================ PROFIL: GAMER ================
Spearman TOPSIS vs SAW = 0.9676 (Irisan Top-10: 8/10)
Spearman TOPSIS vs WP  = 0.9845 (Irisan Top-10: 8/10)


,rank,id,brand,type_name,price_idr,ram_gb,storage_gb,refresh_rate_hz,topsis_score
0,1,240,Acer,Gaming Laptop,32298100,32,1000,300,0.721575
1,2,136,Acer,Gaming Laptop,27548100,16,1000,360,0.674771
2,3,17,MSI,Gaming Laptop,30071300,16,1000,240,0.655742
3,4,643,MSI,Gaming Laptop,34198100,16,1000,240,0.654845
4,5,239,Acer,Gaming Laptop,26598100,16,1000,300,0.647590



================ PROFIL: CREATOR ================
Spearman TOPSIS vs SAW = 0.9224 (Irisan Top-10: 8/10)
Spearman TOPSIS vs WP  = 0.9859 (Irisan Top-10: 8/10)


,rank,id,brand,type_name,price_idr,ram_gb,storage_gb,refresh_rate_hz,topsis_score
0,1,240,Acer,Gaming Laptop,32298100,32,1000,300,0.769117
1,2,155,ASUS,Gaming Laptop,57378100,32,1000,60,0.766815
2,3,161,ASUS,Gaming Laptop,50348100,32,2000,60,0.749085
3,4,2,ASUS,Gaming Laptop,43698100,32,1000,60,0.725089
4,5,1,ASUS,Gaming Laptop,44648100,32,1000,60,0.724552



================ PROFIL: OFFICE ================
Spearman TOPSIS vs SAW = 0.3611 (Irisan Top-10: 4/10)
Spearman TOPSIS vs WP  = 0.7744 (Irisan Top-10: 4/10)


,rank,id,brand,type_name,price_idr,ram_gb,storage_gb,refresh_rate_hz,topsis_score
0,1,395,HP,Thin and Light Laptop,15160100,16,1000,60,0.722013
1,2,638,HP,Thin and Light Laptop,12252910,16,512,60,0.716571
2,3,664,HP,Thin and Light Laptop,12521000,16,512,60,0.716040
3,4,192,Lenovo,Thin and Light Laptop,13869620,16,1000,60,0.713203
4,5,326,HP,Notebook,7596200,8,1000,60,0.707037


In [6]:
# === SEL 6: UJI SENSITIVITAS BOBOT (+-10% & +-20%) ===
sensitivity_results = {}
deltas = [-0.20, -0.10, 0.10, 0.20]

for prof in preset_profiles_output:
    slug = prof['slug']
    base_w = np.array([prof['weights'][c] for c in CRITERIA_CODES], dtype=float)
    base_res = compute_topsis(df, base_w)['ranking_df']
    base_top5 = list(base_res.head(5)['id'])
    top_crit_idx = int(np.argmax(base_w))
    top_crit_code = CRITERIA_CODES[top_crit_idx]

    pert_list = []
    for d in deltas:
        mod_w = base_w.copy()
        mod_w[top_crit_idx] = max(0.001, mod_w[top_crit_idx] * (1.0 + d))
        mod_w = mod_w / mod_w.sum()
        mod_res = compute_topsis(df, mod_w)['ranking_df']
        mod_top5 = list(mod_res.head(5)['id'])
        overlap = len(set(base_top5) & set(mod_top5))
        pert_list.append({
            'delta_pct': int(d * 100),
            'top1_same': bool(base_top5[0] == mod_top5[0]),
            'top5_overlap_count': int(overlap),
            'top5_ids': [int(x) for x in mod_top5]
        })
    sensitivity_results[slug] = {
        'perturbed_criterion': top_crit_code,
        'base_top5_ids': [int(x) for x in base_top5],
        'perturbations': pert_list
    }

print('Hasil Uji Sensitivitas (Perturbasi Kriteria Bobot Tertinggi):')
print(json.dumps(sensitivity_results, indent=2))

Hasil Uji Sensitivitas (Perturbasi Kriteria Bobot Tertinggi):
{
  "student": {
    "perturbed_criterion": "C1",
    "base_top5_ids": [
      326,
      499,
      15,
      157,
      160
    ],
    "perturbations": [
      {
        "delta_pct": -20,
        "top1_same": true,
        "top5_overlap_count": 4,
        "top5_ids": [
          326,
          499,
          638,
          157,
          160
        ]
      },
      {
        "delta_pct": -10,
        "top1_same": true,
        "top5_overlap_count": 4,
        "top5_ids": [
          326,
          499,
          157,
          160,
          416
        ]
      },
      {
        "delta_pct": 10,
        "top1_same": true,
        "top5_overlap_count": 3,
        "top5_ids": [
          326,
          499,
          15,
          122,
          429
        ]
      },
      {
        "delta_pct": 20,
        "top1_same": true,
        "top5_overlap_count": 3,
        "top5_ids": [
          326,
          15,
          122

In [7]:
# === SEL 7: EKSPOR FIXTURE & PAKET ZIP UNTUK LARAVEL & STREAMLIT ===
# Buat deterministik 10-laptop fixture untuk PHPUnit Test di Laravel (toleransi 1e-6)
df_sample10 = df.iloc[:10].copy().reset_index(drop=True)
student_w = [preset_profiles_output[0]['weights'][c] for c in CRITERIA_CODES]
sample10_res = compute_topsis(df_sample10, student_w)

topsis_fixture = {
    'criteria': CRITERIA,
    'weights_used': preset_profiles_output[0]['weights'],
    'sample_10_laptops_input': df_sample10[['id', 'name'] + CRITERIA_COLS].to_dict(orient='records'),
    'expected_denom': [round(float(x), 8) for x in sample10_res['denom']],
    'expected_ideal_pos': [round(float(x), 8) for x in sample10_res['ideal_pos']],
    'expected_ideal_neg': [round(float(x), 8) for x in sample10_res['ideal_neg']],
    'expected_ranking': [
        {
            'rank': int(r['rank']),
            'id': int(r['id']),
            'd_plus': round(float(r['d_plus']), 8),
            'd_minus': round(float(r['d_minus']), 8),
            'topsis_score': round(float(r['topsis_score']), 8)
        }
        for _, r in sample10_res['ranking_df'].iterrows()
    ]
}

validation_summary = {
    'total_laptops_evaluated': int(len(df)),
    'criteria': CRITERIA,
    'preset_profiles': preset_profiles_output,
    'profile_rankings_summary': profile_rankings_summary,
    'sensitivity_analysis': sensitivity_results
}

profiles_path = OUTPUT_DIR / 'preset_profiles.json'
fixture_path = OUTPUT_DIR / 'topsis_fixture.json'
summary_path = OUTPUT_DIR / 'mcdm_validation_summary.json'
clean_fixed_path = OUTPUT_DIR / 'laptops_clean.csv'

profiles_path.write_text(json.dumps(preset_profiles_output, indent=2), encoding='utf-8')
fixture_path.write_text(json.dumps(topsis_fixture, indent=2), encoding='utf-8')
summary_path.write_text(json.dumps(validation_summary, indent=2), encoding='utf-8')
df.to_csv(clean_fixed_path, index=False)

zip_path = Path('lappick_colab_02_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(profiles_path, arcname='preset_profiles.json')
    zf.write(fixture_path, arcname='topsis_fixture.json')
    zf.write(summary_path, arcname='mcdm_validation_summary.json')
    zf.write(clean_fixed_path, arcname='laptops_clean.csv')

print('=== COPY RINGKASAN DI BAWAH INI ATAU TARUH lappick_colab_02_outputs.zip DI FOLDER notebooks/ ===')
print(json.dumps({
    'profiles_cr': {p['slug']: p['cr'] for p in preset_profiles_output},
    'spearman_summary': {
        k: {'vs_saw': v['spearman_topsis_vs_saw'], 'vs_wp': v['spearman_topsis_vs_wp'], 'top1': v['top5_recommendations'][0]['name']}
        for k, v in profile_rankings_summary.items()
    }
}, indent=2))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print('ZIP tersimpan di:', zip_path.resolve())

=== COPY RINGKASAN DI BAWAH INI ATAU TARUH lappick_colab_02_outputs.zip DI FOLDER notebooks/ ===
{
  "profiles_cr": {
    "student": 0.024549,
    "gamer": 0.027025,
    "creator": 0.015273,
    "office": 0.018912
  },
  "spearman_summary": {
    "student": {
      "vs_saw": 0.6778,
      "vs_wp": 0.6485,
      "top1": "hp ryzen 5 octa core 3500u - (8 gb/1 tb hdd/windows 10 home) 245 g8 laptop  (15.6 inch, dark ash silver, with ms office)"
    },
    "gamer": {
      "vs_saw": 0.9676,
      "vs_wp": 0.9845,
      "top1": "acer predator helios 300 core i9 11th gen - (32 gb/1 tb ssd/windows 11 home/8 gb graphics/nvidia geforce rtx 3070/300 hz) ph315-54 gaming laptop  (15.6 inch, abyssal black, 2.3 kg)"
    },
    "creator": {
      "vs_saw": 0.9224,
      "vs_wp": 0.9859,
      "top1": "acer predator helios 300 core i9 11th gen - (32 gb/1 tb ssd/windows 11 home/8 gb graphics/nvidia geforce rtx 3070/300 hz) ph315-54 gaming laptop  (15.6 inch, abyssal black, 2.3 kg)"
    },
    "office": {

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>